# Fiber Photometry Event-Aligned Analysis

This notebook demonstrates a high-level workflow for aligning processed photometry to behavioral and task events.

The reusable analysis code lives in `src/`. This notebook focuses on interactive event-aligned visualization.


## 1. Configuration

Specify the repository, photometry root, and session identifiers. The processed `.npz` path is assembled automatically.


In [ ]:
from pathlib import Path

PHOTOMETRY_ROOT = Path(r'Z:\Photometry')
MOUSE = 'DK21'
DATE = '230704'
RUN = 2
CHANNEL = 1
WINDOW_PRE = 5.0
WINDOW_POST = 10.0

SESSION_DIR = PHOTOMETRY_ROOT / MOUSE / f'{MOUSE}_{DATE}'
PROCESSED_SESSION = SESSION_DIR / f'{MOUSE}-{DATE}-{RUN:03d}-processed.npz'

print('Processed session:', PROCESSED_SESSION)


In [ ]:
# ============================================================
# Find repository root and import project modules
# ============================================================

from pathlib import Path
import sys

current_dir = Path.cwd()

PROJECT_ROOT = None

for candidate in [current_dir, *current_dir.parents]:
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find repository root containing 'src'."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )

# Standard packages
import numpy as np
import matplotlib.pyplot as plt
import pynapple as nap

# Project modules
from src import save_sessiondata
from src import pynapple_utils

print("Repository:")
print(PROJECT_ROOT)

print("\nProcessed session:")
print(PROCESSED_SESSION)

## 2. Load the processed session


In [ ]:
# ============================================================
# Load processed session
# ============================================================

session = save_sessiondata.load_session(
    PROCESSED_SESSION
)

data = pynapple_utils.session_to_pynapple(
    session
)

print("Mouse:", session["mouse"])
print("Date:", session["date"])
print("Run:", session["run"])

print("\nAvailable data:")

for key in data:
    print("  ", key)

## 3. Prepare photometry representations

Both processed dF/F and raw 465 fractional fluorescence are retained for comparison.


In [ ]:
dff_time = np.asarray(data[f'dff_ch{CHANNEL}'].t, dtype=float)
dff = np.asarray(data[f'dff_ch{CHANNEL}'].d, dtype=float)
raw_465_time = np.asarray(session[f'photo_time_465_ch{CHANNEL}'], dtype=float)
raw_465 = np.asarray(session[f'photometry_465_ch{CHANNEL}'], dtype=float)
f0 = np.nanmedian(raw_465)
raw_465_fractional = (raw_465 - f0) / f0

print('dF/F samples:', len(dff))
print('Raw 465 samples:', len(raw_465))


## 4. Event-alignment helper

The helper extracts a fixed window around each event and interpolates each trial onto a shared relative-time grid.


In [ ]:
def event_aligned_traces(signal_time, signal, event_times, pre=5.0, post=10.0):
    signal_time = np.asarray(signal_time, dtype=float)
    signal = np.asarray(signal, dtype=float)
    event_times = np.asarray(event_times, dtype=float)
    dt = np.median(np.diff(signal_time))
    relative_time = np.arange(-pre, post + 0.5 * dt, dt)
    valid_events = [e for e in event_times if e - pre >= signal_time[0] and e + post <= signal_time[-1]]
    valid_events = np.asarray(valid_events, dtype=float)
    traces = np.full((len(valid_events), len(relative_time)), np.nan)
    for i, event in enumerate(valid_events):
        traces[i] = np.interp(event + relative_time, signal_time, signal)
    return relative_time, traces, valid_events

def summarize_aligned_traces(traces):
    mean_trace = np.nanmean(traces, axis=0)
    if traces.shape[0] > 1:
        sem_trace = np.nanstd(traces, axis=0, ddof=1) / np.sqrt(traces.shape[0])
    else:
        sem_trace = np.full(traces.shape[1], np.nan)
    return mean_trace, sem_trace


## 5. Cue-aligned dF/F


In [ ]:
cue_times = np.asarray(session['cue_onset'], dtype=float)
cue_time, cue_dff, cue_events = event_aligned_traces(dff_time, dff, cue_times, WINDOW_PRE, WINDOW_POST)
cue_mean, cue_sem = summarize_aligned_traces(cue_dff)
print('Cue events available:', len(cue_times))
print('Cue events analyzed:', len(cue_events))
print('Trace matrix:', cue_dff.shape)


In [ ]:
plt.figure(figsize=(9,5))
plt.plot(cue_time, cue_mean, linewidth=2, label='Mean dF/F')
plt.fill_between(cue_time, cue_mean-cue_sem, cue_mean+cue_sem, alpha=0.25)
plt.axvline(0, linestyle='--')
plt.axhline(0, linestyle=':')
plt.xlabel('Time from cue onset (s)')
plt.ylabel('dF/F')
plt.title(f'Cue-aligned photometry: {MOUSE} {DATE} run {RUN}')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(10,6))
plt.imshow(cue_dff, aspect='auto', extent=[cue_time[0], cue_time[-1], cue_dff.shape[0], 0], interpolation='none')
plt.axvline(0, linestyle='--')
plt.xlabel('Time from cue onset (s)'); plt.ylabel('Trial')
plt.title('Cue-aligned dF/F heatmap')
plt.colorbar(label='dF/F'); plt.tight_layout(); plt.show()


## 6. Solenoid-aligned dF/F


In [ ]:
solenoid_times = np.asarray(session['solenoid_onset'], dtype=float)
sol_time, sol_dff, sol_events = event_aligned_traces(dff_time, dff, solenoid_times, WINDOW_PRE, WINDOW_POST)
sol_mean, sol_sem = summarize_aligned_traces(sol_dff)
print('Solenoid events available:', len(solenoid_times))
print('Solenoid events analyzed:', len(sol_events))


In [ ]:
plt.figure(figsize=(9,5))
plt.plot(sol_time, sol_mean, linewidth=2, label='Mean dF/F')
plt.fill_between(sol_time, sol_mean-sol_sem, sol_mean+sol_sem, alpha=0.25)
plt.axvline(0, linestyle='--'); plt.axhline(0, linestyle=':')
plt.xlabel('Time from solenoid onset (s)'); plt.ylabel('dF/F')
plt.title('Solenoid-aligned photometry'); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(10,6))
plt.imshow(sol_dff, aspect='auto', extent=[sol_time[0], sol_time[-1], sol_dff.shape[0], 0], interpolation='none')
plt.axvline(0, linestyle='--'); plt.xlabel('Time from solenoid onset (s)'); plt.ylabel('Trial')
plt.title('Solenoid-aligned dF/F heatmap'); plt.colorbar(label='dF/F')
plt.tight_layout(); plt.show()


## 7. Lick-aligned dF/F

Individual licks are aligned here. A complementary analysis can align the onset of detected lick bouts.


In [ ]:
lick_times = np.asarray(session['lick_times'], dtype=float)
lick_time, lick_dff, lick_events = event_aligned_traces(dff_time, dff, lick_times, WINDOW_PRE, WINDOW_POST)
lick_mean, lick_sem = summarize_aligned_traces(lick_dff)
print('Licks available:', len(lick_times))
print('Licks analyzed:', len(lick_events))


In [ ]:
plt.figure(figsize=(9,5))
plt.plot(lick_time, lick_mean, linewidth=2, label='Mean dF/F')
plt.fill_between(lick_time, lick_mean-lick_sem, lick_mean+lick_sem, alpha=0.25)
plt.axvline(0, linestyle='--'); plt.axhline(0, linestyle=':')
plt.xlabel('Time from lick (s)'); plt.ylabel('dF/F')
plt.title('Lick-aligned photometry'); plt.legend(); plt.tight_layout(); plt.show()


## 8. Lick-bout aligned dF/F


In [ ]:
bout_times = np.asarray(session['lick_bout_onset'], dtype=float)
bout_time, bout_dff, bout_events = event_aligned_traces(dff_time, dff, bout_times, WINDOW_PRE, WINDOW_POST)
bout_mean, bout_sem = summarize_aligned_traces(bout_dff)
print('Lick bouts available:', len(bout_times))
print('Lick bouts analyzed:', len(bout_events))


In [ ]:
plt.figure(figsize=(9,5))
plt.plot(bout_time, bout_mean, linewidth=2, label='Mean dF/F')
plt.fill_between(bout_time, bout_mean-bout_sem, bout_mean+bout_sem, alpha=0.25)
plt.axvline(0, linestyle='--'); plt.axhline(0, linestyle=':')
plt.xlabel('Time from lick-bout onset (s)'); plt.ylabel('dF/F')
plt.title('Lick-bout-aligned photometry'); plt.legend(); plt.tight_layout(); plt.show()


## 9. Raw 465 versus dF/F

This comparison is useful when assessing whether reference correction changes the apparent event-related response.


In [ ]:
cue_raw_time, cue_raw, cue_raw_events = event_aligned_traces(raw_465_time, raw_465_fractional, cue_times, WINDOW_PRE, WINDOW_POST)
cue_raw_mean, cue_raw_sem = summarize_aligned_traces(cue_raw)

fig, axes = plt.subplots(2, 1, figsize=(9,8), sharex=True)
axes[0].plot(cue_time, cue_mean, linewidth=2); axes[0].fill_between(cue_time, cue_mean-cue_sem, cue_mean+cue_sem, alpha=0.25)
axes[0].axvline(0, linestyle='--'); axes[0].set_ylabel('dF/F'); axes[0].set_title('Cue-aligned processed dF/F')
axes[1].plot(cue_raw_time, cue_raw_mean, linewidth=2); axes[1].fill_between(cue_raw_time, cue_raw_mean-cue_raw_sem, cue_raw_mean+cue_raw_sem, alpha=0.25)
axes[1].axvline(0, linestyle='--'); axes[1].set_xlabel('Time from cue onset (s)'); axes[1].set_ylabel('Fractional 465'); axes[1].set_title('Cue-aligned raw 465')
plt.tight_layout(); plt.show()


## 10. Notes on interpretation

Event-aligned averages describe temporal associations around events; they do not establish causality by themselves.

For group analysis, summarize trials within each mouse/session before calculating group means so mice contribute at the intended statistical level.
